# Data Exploration — First Look at the Source Files

Before writing any pipeline code, I wanted to get a feel for what's actually in the data files. This notebook covers that initial exploration: what's the structure, how do the two sources relate to each other, and what edge cases might trip up the pipeline.

The two files provided for each company group are:
- `family_tree.json` — the corporate hierarchy (all subsidiaries and affiliates)
- `data_blocks.json` — rich metadata about the top-level company

Three company groups are included: **companyA** (Microsoft), **companyB** (Harford Bank), and **companyC** (Bain Capital).

In [1]:
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../TechTaskDEI_II (2026)")

# quick helper to load any of the json files
def load_json(company: str, filename: str) -> dict:
    path = DATA_DIR / company / filename
    with open(path) as f:
        return json.load(f)

## 1. Family Tree — High-Level Structure

Let me start with companyA's family tree and check what's at the top level.

In [2]:
ft_a = load_json("companyA", "family_tree.json")

# quick look at what's at the top level
ft_a.keys()

dict_keys(['transactionDetail', 'inquiryDetail', 'globalUltimateDuns', 'globalUltimateFamilyTreeMembersCount', 'branchesExcludedMembersCount', 'familyTreeMembers', 'links'])

So the tree has 1,299 total members but the file contains 500 (branches are excluded). Worth keeping in mind that the pipeline works with what's in the file, not the full theoretical tree.

Let me look at what a single member record looks like.

In [3]:
members = ft_a["familyTreeMembers"]

print("total members in file:", len(members))
print("fields:", list(members[0].keys()))

total members in file: 500
fields: ['duns', 'primaryName', 'startDate', 'primaryAddress', 'primaryIndustryCode', 'tradeStyleNames', 'corporateLinkage', 'dunsControlStatus', 'numberOfEmployees', 'financials']


## 2. Understanding the Corporate Hierarchy

The `corporateLinkage` field is the key part here — it's what encodes the parent-child relationships. Let me see how it looks for the root vs a subsidiary.

In [4]:
# root company — no parent
root = members[0]
print("Root company:", root["primaryName"])
print("Hierarchy level:", root["corporateLinkage"]["hierarchyLevel"])
print("Parent:", root["corporateLinkage"].get("parent"))
print("Number of children:", len(root["corporateLinkage"].get("children", [])))

Root company: Microsoft Corporation
Hierarchy level: 1
Parent: None
Number of children: 193


In [5]:
# find a subsidiary — look for one with a parent
subsidiary = next(m for m in members if m["corporateLinkage"].get("parent"))

print("Subsidiary:", subsidiary["primaryName"])
print("Hierarchy level:", subsidiary["corporateLinkage"]["hierarchyLevel"])
print("Parent DUNS:", subsidiary["corporateLinkage"]["parent"]["duns"])
print("Roles:", [r["description"] for r in subsidiary["corporateLinkage"]["familytreeRolesPlayed"]])

Subsidiary: MICROSOFT JAPAN CO., LTD.
Hierarchy level: 2
Parent DUNS: 081466849
Roles: ['Subsidiary', 'Domestic Ultimate', 'Parent/Headquarters']


In [6]:
# what roles appear in the data and how often?
from collections import Counter

roles = Counter()
for m in members:
    for r in m["corporateLinkage"].get("familytreeRolesPlayed", []):
        roles[r["description"]] += 1

print("Role distribution (companyA):")
for role, count in roles.most_common():
    print(f"  {role}: {count}")

Role distribution (companyA):
  Subsidiary: 483
  Domestic Ultimate: 295
  Parent/Headquarters: 128
  Branch/Division: 16
  Global Ultimate: 1


In [7]:
# hierarchy depth — how many levels?
levels = Counter(m["corporateLinkage"].get("hierarchyLevel") for m in members)
print("Members per hierarchy level:")
for level in sorted(levels):
    print(f"  Level {level}: {levels[level]} companies")

Members per hierarchy level:
  Level 1: 1 companies
  Level 2: 193 companies
  Level 3: 142 companies
  Level 4: 43 companies
  Level 5: 12 companies
  Level 6: 67 companies
  Level 7: 38 companies
  Level 8: 4 companies


8 levels deep. That's important for the schema — the parent-child relationship needs to work recursively, not just one level down.

Also notice that a company can have **multiple roles at once** (Activision Blizzard is both a Subsidiary and a Parent/HQ). That means a single `role` column won't cut it — roles need their own table.

## 3. Data Quality Check — What's Missing?

Before designing the schema, I want to know which fields can be null or empty.

In [8]:
df = pd.DataFrame(members)

# check which fields have nulls
df.isnull().sum()[df.isnull().sum() > 0]

startDate               14
primaryIndustryCode      1
tradeStyleNames        410
numberOfEmployees       59
financials              20
dtype: int64

In [9]:
# tradeStyleNames stands out — let me check more carefully
# pandas converts JSON null → NaN (float), so we can't use `x is None`
def classify_field(x):
    if not isinstance(x, list):
        return "null"
    if len(x) == 0:
        return "empty list"
    return "has values"

trade_style_states = df["tradeStyleNames"].apply(classify_field).value_counts()
print("tradeStyleNames breakdown:")
print(trade_style_states)

tradeStyleNames breakdown:
tradeStyleNames
null          410
has values     90
Name: count, dtype: int64


Interesting — `tradeStyleNames` is actually `null` (not an empty list) for most members. That means I can't just do `member.get("tradeStyleNames", [])` — I need to handle the explicit null too: `member.get("tradeStyleNames") or []`.

Let me also check `numberOfEmployees` and `financials`, since they're lists.

In [10]:
# check if these list fields are ever empty lists vs null
# note: pandas stores JSON null as NaN (float), so we check isinstance first
for field in ["numberOfEmployees", "financials"]:
    def classify(x):
        if not isinstance(x, list):
            return "null / NaN"
        if len(x) == 0:
            return "empty list"
        return f"has {len(x)} item(s)"

    states = df[field].apply(classify).value_counts()
    print(f"\n{field}:")
    print(states)


numberOfEmployees:
numberOfEmployees
has 1 item(s)    441
null / NaN        59
Name: count, dtype: int64

financials:
financials
has 1 item(s)    480
null / NaN        20
Name: count, dtype: int64


In [11]:
# startDate has mixed formats — year only vs full date
df["startDate"].dropna().unique()[:15]

<ArrowStringArray>
[      '1975',    '1986-02',       '1979',       '2019', '1983-04-01',
       '2003', '1988-07-22',       '1995',       '1998',       '1988',
 '1997-04-02', '1992-09-02',       '2007',       '1985', '2011-09-26']
Length: 15, dtype: str

Confirmed: `startDate` can be just a year (`"1975"`) or a full date (`"1993-09-22"`). Storing it as VARCHAR in the schema makes more sense than trying to force a DATE type.

## 4. The data_blocks File — What Does It Add?

Now let me look at `data_blocks.json`. My hypothesis is that it's much richer but only covers the top-level company.

In [12]:
db_a = load_json("companyA", "data_blocks.json")

print("DUNS:", db_a["duns"])
print("Company:", db_a["primaryName"])
print()
print(f"Number of fields: {len(db_a)}")
print()
print("All fields available:")
for k in db_a.keys():
    v = db_a[k]
    if isinstance(v, list):
        print(f"  {k}: list ({len(v)} items)")
    elif isinstance(v, dict):
        print(f"  {k}: dict")
    else:
        print(f"  {k}: {v}")

DUNS: 081466849
Company: Microsoft Corporation

Number of fields: 63

All fields available:
  duns: 081466849
  isFortune1000Listed: True
  isForbesLargestPrivateCompaniesListed: None
  industryCodes: list (22 items)
  isNonClassifiedEstablishment: None
  primaryIndustryCode: dict
  primaryName: Microsoft Corporation
  controlOwnershipDate: 1975
  isStandalone: False
  telephone: list (1 items)
  isAgent: None
  isImporter: None
  isExporter: None
  isSmallBusiness: None
  organizationSizeCategory: dict
  tradeStyleNames: list (1 items)
  registeredAddress: dict
  primaryAddress: dict
  businessEntityType: dict
  countryISOAlpha2Code: US
  dunsControlStatus: dict
  controlOwnershipType: dict
  financials: list (1 items)
  summary: list (8 items)
  multilingualPrimaryName: list (0 items)
  multilingualRegisteredNames: list (0 items)
  legalForm: dict
  multilingualTradestyleNames: list (0 items)
  preferredLanguage: dict
  stockExchanges: list (30 items)
  mailingAddress: dict
  website

In [13]:
# check the financials
financials = db_a.get("financials", [])
if financials:
    f = financials[0]
    print("Latest financial statement:")
    print("  Date:", f.get("financialStatementToDate"))
    print("  Scope:", f.get("informationScopeDescription"))
    print("  Reliability:", f.get("reliabilityDescription"))
    revenues = f.get("yearlyRevenue", [])
    if revenues:
        print(f"  Revenue: ${revenues[0]['value']:,.0f} {revenues[0]['currency']}")

Latest financial statement:
  Date: 2024-06-30
  Scope: Consolidated
  Reliability: Actual
  Revenue: $245,122,000,000 USD


In [14]:
# employees
employees = db_a.get("numberOfEmployees", [])
if employees:
    e = employees[0]
    print("Employees:")
    print("  Count:", e.get("value"))
    print("  Scope:", e.get("informationScopeDescription"))
    print("  Reliability:", e.get("reliabilityDescription"))

Employees:
  Count: 228000
  Scope: Consolidated
  Reliability: Actual


In [15]:
# industry codes — how many and what types?
industry_codes = db_a.get("industryCodes", [])
print(f"Total industry codes: {len(industry_codes)}")
print()
code_types = Counter(c["typeDescription"] for c in industry_codes)
print("Classification systems used:")
for system, count in code_types.items():
    print(f"  {system}: {count} codes")

Total industry codes: 22

Classification systems used:
  North American Industry Classification System 2022: 6 codes
  D&B Standard Industry Code: 6 codes
  NACE Revision 2: 1 codes
  US Standard Industry Code 1987 - 4 digit: 6 codes
  D&B Hoovers Industry Classification: 1 codes
  D&B Standard Major Industry Code: 1 codes
  ISIC Revision 4: 1 codes


22 industry codes across multiple classification systems. This confirms that `industry_codes` needs to be a separate table with a `type_description` field — there's no way to put this cleanly in a single row.

## 5. How Do the Two Files Join?

The DUNS from `data_blocks.json` should match the Global Ultimate DUNS in `family_tree.json`. Let me confirm.

In [16]:
print("family_tree globalUltimateDuns:", ft_a["globalUltimateDuns"])
print("data_blocks duns:              ", db_a["duns"])
print("Match:", ft_a["globalUltimateDuns"] == db_a["duns"])
print()

# so the join is: for each member of the family tree,
# the data_blocks file enriches the group with the global ultimate's data
root_member = next(m for m in members if m["duns"] == ft_a["globalUltimateDuns"])
print("Root in family tree:", root_member["primaryName"])
print("Root in data_blocks:", db_a["primaryName"])

family_tree globalUltimateDuns: 081466849
data_blocks duns:               081466849
Match: True

Root in family tree: Microsoft Corporation
Root in data_blocks: Microsoft Corporation


In [17]:
# quick prototype of the enrichment logic
# for each member, extract: parent_duns, hierarchy_level, global_ultimate_duns

def extract_member_fields(member: dict, global_duns: str) -> dict:
    linkage = member.get("corporateLinkage") or {}
    parent = linkage.get("parent") or {}
    roles = linkage.get("familytreeRolesPlayed") or []
    address = member.get("primaryAddress") or {}

    return {
        "duns": member.get("duns"),
        "primary_name": member.get("primaryName"),
        "country_iso": (address.get("addressCountry") or {}).get("isoAlpha2Code"),
        "city": (address.get("addressLocality") or {}).get("name"),
        "parent_duns": parent.get("duns"),           # None for root
        "global_ultimate_duns": global_duns,
        "hierarchy_level": linkage.get("hierarchyLevel"),
        "roles": ", ".join(r["description"] for r in roles),
    }

rows = [extract_member_fields(m, ft_a["globalUltimateDuns"]) for m in members]
df_enriched = pd.DataFrame(rows)

print(df_enriched.head(10).to_string())

        duns                                   primary_name country_iso                 city parent_duns global_ultimate_duns  hierarchy_level                                                    roles
0  081466849                          Microsoft Corporation          US              Redmond         NaN            081466849                1  Global Ultimate, Domestic Ultimate, Parent/Headquarters
1  690763115                      MICROSOFT JAPAN CO., LTD.          JP            MINATO-KU   081466849            081466849                2       Subsidiary, Domestic Ultimate, Parent/Headquarters
2  098533342                      Activision Blizzard, Inc.          US         Santa Monica   081466849            081466849                2                          Subsidiary, Parent/Headquarters
3  659563882             MICROSOFT REGIONAL SALES PTE. LTD.          SG            Singapore   081466849            081466849                2       Subsidiary, Domestic Ultimate, Parent/Headquarters


In [18]:
# spot check: root company should have parent_duns = NaN
root_row = df_enriched[df_enriched["duns"] == ft_a["globalUltimateDuns"]].iloc[0]
print("Root company parent_duns:", root_row["parent_duns"])
print("Is null?", pd.isna(root_row["parent_duns"]))
print()

# subsidiaries should have a parent_duns
subs = df_enriched[df_enriched["parent_duns"].notna()]
print(f"Members with a parent: {len(subs)} out of {len(df_enriched)}")

Root company parent_duns: nan
Is null? True

Members with a parent: 499 out of 500


## 6. Comparing the Three Companies

The three datasets are intentionally very different. Let me get a quick overview of each to make sure the pipeline handles all cases.

In [19]:
summary = []

for company in ["companyA", "companyB", "companyC"]:
    ft = load_json(company, "family_tree.json")
    db = load_json(company, "data_blocks.json")
    members_local = ft["familyTreeMembers"]

    employees = db.get("numberOfEmployees", [])
    emp_count = employees[0]["value"] if employees else None

    financials_local = db.get("financials", [])
    revenue = None
    if financials_local:
        rev_list = financials_local[0].get("yearlyRevenue", [])
        revenue = rev_list[0]["value"] if rev_list else None

    max_level = max(
        (m["corporateLinkage"].get("hierarchyLevel") or 0) for m in members_local
    )

    summary.append({
        "company": db["primaryName"],
        "entity_type": db.get("businessEntityType", {}).get("description", "N/A"),
        "members_in_file": len(members_local),
        "total_in_tree": ft["globalUltimateFamilyTreeMembersCount"],
        "max_depth": max_level,
        "employees": emp_count,
        "revenue_usd": revenue,
    })

pd.DataFrame(summary).set_index("company")

,entity_type,members_in_file,total_in_tree,max_depth,employees,revenue_usd
company,,,,,,
Microsoft Corporation,Corporation,500,1299,8,228000,2.451220e+11
Harford Bank,Corporation,1,9,1,35,3.028200e+07
"Bain Capital, LP",Partnership,375,1128,11,500,1.803900e+06


Three very different cases:
- **Microsoft** — large public corporation, deep hierarchy (8 levels), full financial data
- **Harford Bank** — tiny regional bank, 9 members, minimal data
- **Bain Capital** — large but structured as a Partnership (LP), not a Corporation

The pipeline and schema need to handle all three without special-casing any of them.

In [20]:
# check: does Harford Bank (companyB) have any financials in data_blocks?
db_b = load_json("companyB", "data_blocks.json")
print("Harford Bank financials:", db_b.get("financials", []))
print("Harford Bank employees:", db_b.get("numberOfEmployees", []))

Harford Bank financials: [{'financialStatementToDate': '2023-12-31', 'financialStatementDuration': None, 'informationScopeDescription': 'Individual', 'informationScopeDnBCode': 9066, 'reliabilityDescription': 'Actual', 'reliabilityDnBCode': 9092, 'unitCode': 'SingleUnits', 'yearlyRevenue': [{'value': 30282000.0, 'currency': 'USD'}]}]
Harford Bank employees: [{'value': 35, 'employeeFiguresDate': None, 'informationScopeDescription': 'Headquarters Only (Employs Here)', 'informationScopeDnBCode': 9068, 'reliabilityDescription': 'Actual', 'reliabilityDnBCode': 9092, 'employeeCategories': []}, {'value': 80, 'employeeFiguresDate': None, 'informationScopeDescription': 'Consolidated', 'informationScopeDnBCode': 9067, 'reliabilityDescription': 'Actual', 'reliabilityDnBCode': 9092, 'employeeCategories': [{'employmentBasisDescription': 'Principals', 'employmentBasisDnBCode': 9064}], 'trend': []}]


Harford Bank has no financial data — confirms that `global_revenue_usd` and `global_employee_count` need to handle empty lists gracefully.

## 7. Address Fields

The `data_blocks` file has multiple address types. Let me check what's available.

In [21]:
address_types = ["primaryAddress", "registeredAddress", "mailingAddress"]

for addr_type in address_types:
    addr = db_a.get(addr_type, {})
    if addr:
        city = (addr.get("addressLocality") or {}).get("name", "N/A")
        country = (addr.get("addressCountry") or {}).get("isoAlpha2Code", "N/A")
        lat = addr.get("latitude")
        print(f"{addr_type}: {city}, {country} | lat: {lat}")
    else:
        print(f"{addr_type}: (not present)")

primaryAddress: Redmond, US | lat: 47.63921
registeredAddress: (not present)
mailingAddress: (not present)


Three distinct address types for Microsoft, and the primary address has geocoordinates. Good reason to keep all three in a single `addresses` table with an `address_type` discriminator — cleaner than three separate address columns on the company record.

## 8. Key Takeaways

After going through the data, here's what shaped the schema and pipeline design:

**Schema decisions driven by the data:**
1. DUNS is the natural primary key — unique, immutable, present in both files
2. Corporate hierarchy = self-referencing FK (`parent_duns → companies.duns`)
3. `hierarchy_roles` needs its own table — a company can have multiple roles simultaneously
4. `industry_codes` needs its own table — up to 22 codes in 4 different classification systems
5. `addresses` in one table with `address_type` — avoids null columns and handles 0-3 addresses cleanly
6. `information_scope` and `reliability` must be stored with financials/employees — they change the meaning of the number

**Pipeline decisions driven by the data:**
1. `tradeStyleNames` can be `null` (not empty list) — need `or []` not just `.get("field", [])`
2. `startDate` has two formats — store as VARCHAR, don't try to parse as DATE
3. `numberOfEmployees` and `financials` can be empty lists — check length before accessing index 0
4. Join between files is a **left join** — `data_blocks` only covers the Global Ultimate; subsidiaries still need to appear in the output
5. Schema must work for tiny (Harford Bank, 9 members) and large (Microsoft, 500+ members) equally well